# Pricing Mushroom base agent

> Base pricing agent for the integration of mushroom_rl-based agents

In [ ]:
#| default_exp agents.dynamic_pricing.mushroom_rl

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

# set logging level to INFO
logging.basicConfig(level=logging.INFO)

from abc import ABC, abstractmethod
from typing import Union, Optional, List, Tuple
import numpy as np
import os

from ddopai.agents.base import BaseAgent
from ddopai.utils import MDPInfo, Parameter

import torch
import torch.nn.functional as F

import time

In [ ]:
#| export

class PricingMushroomBaseAgent(BaseAgent):

    """
    Base class for Agents that integrate MushroomRL agents.
    """

    train_mode = "env_interaction"
    dropout = True # always keep in True for mushroom_RL, dropout is not desired set drop_prob=0.0
    
    def __init__(self, 
            environment_info: MDPInfo,
            obsprocessors: Optional[List] = None,     # default: []
            device: str = "cpu", # "cuda" or "cpu"
            agent_name: str | None = None
            ):

        self.device = device

        super().__init__(environment_info = environment_info, obsprocessors = obsprocessors, agent_name = agent_name)

        self.transfer_obs_processors_to_mushroom_agent()

    def transfer_obs_processors_to_mushroom_agent(self):
    
        """ Transfer the obs-processors to the MushroomRL agent preprocessors"""

        for obsprocessor in self.obsprocessors:
            self.add_obsprocessor(obsprocessor)
        self.obsprocessors = []

    def add_obsprocessor(self, obsprocessor: object): 
        """Add an obsprocessor to the agent - overwrites the base
        class method to add the obsprocessor to the MushroomRL agent
        as preprocessor"""
        self.agent.add_preprocessor(obsprocessor)

    @property
    def preprocessors(self):

        """ Return the obsprocessors of the agent,
        which are the preprocessors of the MushroomRL agent """

        return self.agent.preprocessors

    def draw_action_(self, observation: np.ndarray) -> np.ndarray: #
        
        """ 
        Draw an action based on the fitted model (see predict method)
        """

        observation = self.remove_batch_dim(observation)
    
        if self.mode=="train":
            action = self.agent.draw_action(observation)
        else:
            action = self.agent.draw_action(observation) #action = self.predict(observation) # bypass the agent's draw_action method and directly get prediction from policy network
        
        return action

    def train(self):
        """set the internal state of the agent and its model to train"""
        self.mode = "train"
        

    def eval(self):
        """set the internal state of the agent and its model to eval"""
        self.mode = "eval"
        
    
    def to(self, device: str): #
        """Move the model to the specified device"""

        # check if self.model or something else
        self.model.to(device)

    def save(self,
                path: str, # The directory where the file will be saved.
                overwrite: bool=True): # Allow overwriting; if False, a FileExistsError will be raised if the file exists.
        
        """
        Save the PyTorch model to a file in the specified directory.

        """
        
        if not hasattr(self, 'network_list') or self.network_list is None:
            raise AttributeError("Cannot find networks.")

        # Create the directory path if it does not exist
        os.makedirs(path, exist_ok=True)

        # Construct the file path using os.path.join for better cross-platform compatibility

        for network_number, network in enumerate(self.network_list):
            full_path = os.path.join(path, f"network_{network_number}.pth")

            if os.path.exists(full_path):
                if not overwrite:
                    raise FileExistsError(f"The file {full_path} already exists and will not be overwritten.")
                else:
                    logging.debug(f"Overwriting file {full_path}") # Only log with info as during training we will continuously overwrite the model
            
            # Save the model's state_dict using torch.save
            torch.save(network.state_dict(), full_path)
        logging.debug(f"Model saved successfully to {full_path}")

    def load(self, path: str):
        """
        Load the PyTorch models from files in the specified directory.
        """
        
        if not hasattr(self, 'network_list') or self.network_list is None:
            raise AttributeError("Cannot find networks to load.")

        # Check for the presence of model files
        for network_number, network in enumerate(self.network_list):
            full_path = os.path.join(path, f"network_{network_number}.pth")

            if not os.path.exists(full_path):
                raise FileNotFoundError(f"The file {full_path} does not exist.")
            
            try:
                # Load each network's state_dict
                network.load_state_dict(torch.load(full_path))
                logging.info(f"Network {network_number} loaded successfully from {full_path}")
            except Exception as e:
                raise RuntimeError(f"An error occurred while loading network {network_number}: {e}")

    def set_device(self, device: str):

        """ Set the device for the model """

        if device == "cuda":
            if torch.cuda.is_available():
                use_cuda = True
            else:
                logging.warning("CUDA is not available. Using CPU instead.")
                use_cuda = False
        elif device == "cpu":
            use_cuda = False
        else:
            raise ValueError(f"Device {device} not currently not supported, use 'cuda' or 'cpu'")

        return use_cuda

    def episode_start(self):

        """ What to do if a new episode starts, e.g., reset policy of the agent
        Often this does not need to do anything (default), otherwise this funciton 
        needs to be overwritten in the subclass. """

        pass

    def fit(self, dataset, **dataset_info):

        """ Hand the fit mehtod to the mushroom agent """

        self.agent.fit(dataset, **dataset_info)

    def stop(self):
        """ Stop the agent """

        self.agent.stop()

    import numpy as np

    @staticmethod
    def remove_batch_dim(input: np.ndarray | dict[str, np.ndarray]) -> np.ndarray | dict[str, np.ndarray]: #
        
        """
        Remove the batch dimension from the input array or from each value in the dictionary of arrays.
        Assumes the batch dimension is the first dimension (axis=0).
        
        """
        
        # Check if the input is a numpy array or a dict of numpy arrays
        if isinstance(input, np.ndarray):
            # Remove the batch dimension by squeezing the first axis (axis=0)
            return np.squeeze(input, axis=0)
        elif isinstance(input, dict):
            # Remove the batch dimension from each numpy array in the dict
            return {key: np.squeeze(arr, axis=0) for key, arr in input.items()}
        else:
            raise TypeError("Input must be a numpy array or a dictionary of numpy arrays.")
